In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Load the merged dataset
df = pd.read_csv('/Users/yaswanthkanagarla/Desktop/Master_Thesis/BD_code/Battery_Diagnostics_Thesis/scripts/final_7features_state_IX.csv')

state_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9}
df['state'] = df['state'].map(state_map)


In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import mean_squared_error

feature_cols = [
    "R_s_rel","R_ct_rel","R_sei_rel","Zmag_0.1Hz_rel",
    "C_dl_log","tail_slope","Zmag_1000Hz_rel","T_C"
]

target_col = "SOH"

df["group_ct"] = df["T_C"].astype(str) + "_cell" + df["cell"].astype(str)

# state selection
df9 = df[df["state"] == 9].copy()

# clean inf/nan
df9[feature_cols] = df9[feature_cols].replace([np.inf, -np.inf], np.nan)
df9 = df9.dropna(subset=feature_cols + [target_col]).copy()

X = df9[feature_cols]
y = df9[target_col]
groups = df9["group_ct"]

def metrics(y_true, y_pred):
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

def make_rf():
    return RandomForestRegressor(
        n_estimators=800,
        random_state=42,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2
    )


groupkfolds

In [11]:
gkf = GroupKFold(n_splits=4)

rows = []
for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups), start=1):
    X_train, X_val = X.iloc[tr], X.iloc[va]
    y_train, y_val = y.iloc[tr], y.iloc[va]

    model = make_rf()
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    m = metrics(y_val, pred)

    held_out = groups.iloc[va].unique().tolist()
    rows.append({
        "fold": fold,
        "n_val_groups": len(held_out),
        "val_groups": ", ".join(held_out),
        **m,
        "n_train": len(tr),
        "n_val": len(va)
    })

gkf_df = pd.DataFrame(rows)
print("GroupKFold (4 folds)")
print("R2   mean±std:", gkf_df["R2"].mean(), "±", gkf_df["R2"].std())
print("MAE  mean±std:", gkf_df["MAE"].mean(), "±", gkf_df["MAE"].std())
print("RMSE mean±std:", gkf_df["RMSE"].mean(), "±", gkf_df["RMSE"].std())

gkf_df.sort_values("R2")


GroupKFold (4 folds)
R2   mean±std: 0.7702347362030831 ± 0.07475276711210303
MAE  mean±std: 0.0607239364096101 ± 0.03038485561983779
RMSE mean±std: 0.0742749940088364 ± 0.03737541368481404


,fold,n_val_groups,val_groups,R2,MAE,RMSE,n_train,n_val
2,3,3,"25_cell8, 35_cell1, 45_cell1",0.689700,0.050733,0.063032,2074,662
1,2,3,"25_cell2, 25_cell5, 25_cell6",0.747367,0.088400,0.113313,2016,720
0,1,3,"25_cell1, 25_cell3, 25_cell7",0.774983,0.081441,0.093196,2045,691
3,4,3,"25_cell4, 35_cell2, 45_cell2",0.868889,0.022321,0.027559,2073,663


loco

In [4]:
logo = LeaveOneGroupOut()

rows = []
for fold, (tr, va) in enumerate(logo.split(X, y, groups=groups), start=1):
    X_train, X_val = X.iloc[tr], X.iloc[va]
    y_train, y_val = y.iloc[tr], y.iloc[va]

    model = make_rf()
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    m = metrics(y_val, pred)

    held_out = groups.iloc[va].unique().tolist()
    rows.append({
        "fold": fold,
        "held_out_group": held_out[0] if len(held_out) == 1 else str(held_out),
        **m,
        "n_train": len(tr),
        "n_val": len(va)
    })

loco_df = pd.DataFrame(rows)
print("LOCO (Leave-One-(T_C,cell)-Out)")
print("R2   mean±std:", loco_df["R2"].mean(), "±", loco_df["R2"].std())
print("MAE  mean±std:", loco_df["MAE"].mean(), "±", loco_df["MAE"].std())
print("RMSE mean±std:", loco_df["RMSE"].mean(), "±", loco_df["RMSE"].std())

loco_df.sort_values("R2").head(10)


LOCO (Leave-One-(T_C,cell)-Out)
R2   mean±std: -0.724369941927082 ± 2.714751189520361
MAE  mean±std: 0.06405004162408105 ± 0.035118958329760225
RMSE mean±std: 0.07313952046968963 ± 0.03862975140569325


,fold,held_out_group,R2,MAE,RMSE,n_train,n_val
2,3,25_cell3,-7.817826,0.122271,0.123588,2534,202
3,4,25_cell4,-4.563905,0.063395,0.066860,2701,35
1,2,25_cell2,-1.237961,0.045124,0.049032,2555,181
7,8,25_cell8,-0.289317,0.048250,0.052859,2699,37
5,6,25_cell6,0.210229,0.061978,0.083883,2524,212
10,11,45_cell1,0.415106,0.043087,0.051793,2437,299
6,7,25_cell7,0.602867,0.074151,0.084398,2596,140
8,9,35_cell1,0.676020,0.058912,0.073141,2410,326
4,5,25_cell5,0.704792,0.139203,0.161869,2409,327
11,12,45_cell2,0.853144,0.022416,0.026432,2426,310


In [23]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    }

In [ ]:

# Load the merged dataset
df = pd.read_csv('/Users/yaswanthkanagarla/Desktop/Master_Thesis/BD_code/Battery_Diagnostics_Thesis/scripts/merged_eis_capacity_state_IX_norm.csv')
state_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9}
df['state'] = df['state'].map(state_map)
train_groups = [(25, 6), (25, 7), (25, 8), (25, 5), (25, 3), (25, 4), (35, 1), (25, 2), (45, 2)]
val_groups   = []                 # validation cell(s)
test_groups  = [(25, 1), (35, 2), (45, 1)]         # final test cell(s)

In [25]:
def mask_from_groups(df, groups):
    groups = set(groups)
    return df.apply(lambda r: (r["T_C"], r["cell"]) in groups, axis=1)

train_mask = mask_from_groups(df, train_groups)
#val_mask   = mask_from_groups(df, val_groups)
test_mask  = mask_from_groups(df, test_groups)

train_df = df[train_mask].copy()
#val_df   = df[val_mask].copy()
test_df  = df[test_mask].copy()

print("Train:", train_df.shape, "Groups:", sorted(set(zip(train_df.T_C, train_df.cell))))
#print("Val  :", val_df.shape,   "Groups:", sorted(set(zip(val_df.T_C, val_df.cell))))
print("Test :", test_df.shape,  "Groups:", sorted(set(zip(test_df.T_C, test_df.cell))))



Train: (1770, 31) Groups: [(25, 2), (25, 3), (25, 4), (25, 5), (25, 6), (25, 7), (25, 8), (35, 1), (45, 2)]
Test : (966, 31) Groups: [(25, 1), (35, 2), (45, 1)]


In [30]:
features = [
    "R_s_rel",
    "R_ct_rel",
    "R_sei_rel",
    "Zmag_0.1Hz_rel",
    "C_dl_log",
    "tail_slope",
    "Zmag_1000Hz_rel",
    "T_C"
]

X_train = train_df[features].copy()
y_train = train_df["SOH"].copy()

X_test  = test_df[features].copy()
y_test  = test_df["SOH"].copy()

for X in (X_train, X_test):
    X.replace([np.inf, -np.inf], np.nan, inplace=True)

from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median")

X_train[:] = imp.fit_transform(X_train)
X_test[:]  = imp.transform(X_test)

print("NaNs train:", X_train.isna().sum().sum(),
      "test:", X_test.isna().sum().sum())



NaNs train: 0 test: 0


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MSE": mean_squared_error(y_true, y_pred),
    }

#  groups for TRAIN only (cell + temp combo)
groups_train = df.loc[X_train.index, ["cell", "T_C"]].astype(str).agg("_".join, axis=1)
X_train_feat = X_train.drop(columns=["cell"]) if "cell" in X_train.columns else X_train
X_test_feat  = X_test.drop(columns=["cell"])  if "cell" in X_test.columns else X_test


In [ ]:
def evaluate_model_groupcv_and_test(model, X_train, y_train, groups_train, X_test, y_test, n_splits=5):
    
    n_splits = min(n_splits, pd.Series(groups_train).nunique())
    gkf = GroupKFold(n_splits=n_splits)

    oof_pred = np.zeros(len(X_train))


    for tr_idx, val_idx in gkf.split(X_train, y_train, groups=groups_train):
        m = clone(model)
        m.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        oof_pred[val_idx] = m.predict(X_train.iloc[val_idx])

    cv_scores = metrics(y_train, oof_pred)

   
    m_full = clone(model)
    m_full.fit(X_train, y_train)
    test_pred = m_full.predict(X_test)

    test_scores = metrics(y_test, test_pred)

    return cv_scores, test_scores, test_pred


In [33]:
from sklearn.ensemble import RandomForestRegressor

rf_base = RandomForestRegressor(
    n_estimators=600,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=1
)

rf_base.fit(X_train, y_train)
rf_base.fit(X_train, y_train)
print("RF TEST:", metrics(y_test, rf_base.predict(X_test)))

RF TEST: {'R2': 0.8955195004457781, 'MAE': 0.04853078421423649, 'RMSE': np.float64(0.05868915338290508), 'MSE': 0.003444416724802158}


In [ ]:
import numpy as np
from xgboost import XGBRegressor



xgb_stack = XGBRegressor(
    n_estimators=8000,
    learning_rate=0.08,
    max_depth=3,
    subsample=1.0,
    colsample_bytree=0.7,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror",
    eval_metric="rmse"
   
)
xgb_stack.fit(X_train, y_train)
print("XGB TEST:", metrics(y_test, xgb_stack.predict(X_test)))

XGB TEST: {'R2': 0.8696147884407313, 'MAE': 0.04972599613031385, 'RMSE': np.float64(0.06556233196569602), 'MSE': 0.004298419372780127}


In [35]:
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=-1,          # -1 = no limit
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    min_child_samples=50,
    random_state=42,
    n_jobs=-1,
    #force_col_wise=true
)
lgbm.fit(X_train, y_train)
print("LGBM TEST:", metrics(y_test, lgbm.predict(X_test)))



[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000215 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1789
[LightGBM] [Info] Number of data points in the train set: 1770, number of used features: 8
[LightGBM] [Info] Start training from score 0.730683
LGBM TEST: {'R2': 0.8539953426742571, 'MAE': 0.04881568730676375, 'RMSE': np.float64(0.06937828804233398), 'MSE': 0.0048133468516850615}


In [36]:
from catboost import CatBoostRegressor

cat = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    verbose=0,
    random_seed=42
)

cat.fit(X_train, y_train)

print("Train:", metrics(y_train, cat.predict(X_train)))
#print("Val  :", metrics(y_val,   cat.predict(X_val)))
print("Test :", metrics(y_test,  cat.predict(X_test)))

Train: {'R2': 0.9991527766893136, 'MAE': 0.003914270672870622, 'RMSE': np.float64(0.00517756591891994), 'MSE': 2.6807188844761286e-05}
Test : {'R2': 0.889995898175617, 'MAE': 0.04509748708282488, 'RMSE': np.float64(0.060220542581881045), 'MSE': 0.0036265137488561486}


In [ ]:
base_models = {"rf": rf_base, "xgb": xgb_stack, "lgb": lgbm, "cb": cat}

In [38]:
results = []
test_preds_single = {}

for name, model in base_models.items():
    cv_scores, test_scores, test_pred = evaluate_model_groupcv_and_test(
        model,
        X_train_feat, y_train,
        groups_train,
        X_test_feat, y_test,
        n_splits=5
    )

    results.append({
        "model": name,
        "CV_R2": cv_scores["R2"],
        "CV_RMSE": cv_scores["RMSE"],
        "CV_MAE": cv_scores["MAE"],
        "TEST_R2": test_scores["R2"],
        "TEST_RMSE": test_scores["RMSE"],
        "TEST_MAE": test_scores["MAE"],
    })

    test_preds_single[name] = test_pred

results_df = pd.DataFrame(results).sort_values("TEST_R2", ascending=False)
results_df


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000118 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1789
[LightGBM] [Info] Number of data points in the train set: 1443, number of used features: 8
[LightGBM] [Info] Start training from score 0.768284
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000196 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1788
[LightGBM] [Info] Number of data points in the train set: 1409, number of used features: 8
[LightGBM] [Info] Start training from score 0.724022
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000171 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1788
[LightGBM] [Info] Number of data points in the train set: 1423, number of used features: 8
[LightGBM] [Info] Start training 

,model,CV_R2,CV_RMSE,CV_MAE,TEST_R2,TEST_RMSE,TEST_MAE
0,rf,0.522222,0.122953,0.086624,0.895520,0.058689,0.048531
3,cb,0.404940,0.137217,0.088399,0.889996,0.060221,0.045097
1,xgb,0.454276,0.131405,0.098714,0.869615,0.065562,0.049726
2,lgb,0.521976,0.122985,0.091174,0.853995,0.069378,0.048816


In [ ]:
cv_preds = {}

for name in base_models.keys():
    cv_preds[name] = oof_train[name].values


NameError: name 'oof_train' is not defined